# Vocal Tract Demo

Demonstrates the full Pink Trombone model: glottis + Kelly-Lochbaum waveguide tract.
The tract shapes the raw glottal buzz into vowels and other sounds via two sets of controls:

- **tongueIndex / tongueDiameter** — tongue body position (12.9 = neutral, lower = back vowels, higher = front vowels; larger diameter = more open)
- **constrictionIndex / constrictionDiameter** — additional constriction (e.g. for fricatives or stop consonants; negative diameter = closed)


In [ ]:
import torch
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from samuel.pink_trombone import (
    pink_trombone,
    PARAM_NAMES,
    N_PARAMS,
    SAMPLE_RATE,
    SAMPLES_PER_FRAME,
    _compute_diameter_profile,
    _TRACT_N,
)

In [ ]:
CONTROL_RATE = SAMPLE_RATE / SAMPLES_PER_FRAME  # 12.5 Hz

def make_params(duration_s=2.0, **overrides):
    """Create a parameter tensor with vocal-tract defaults."""
    T = int(duration_s * CONTROL_RATE)
    defaults = {
        "noise": 0.1,
        "frequency": 140,
        "tenseness": 0.6,
        "intensity": 1,
        "loudness": 1,
        "tongueIndex": 12.9,
        "tongueDiameter": 2.43,
        "vibratoWobble": 0,
        "vibratoFrequency": 6,
        "vibratoGain": 0.0,
        "tractLength": 44,
        "constrictionIndex": 25.0,
        "constrictionDiameter": 2.0,  # open (no constriction effect)
    }
    params = torch.zeros(1, T, N_PARAMS)
    for i, name in enumerate(PARAM_NAMES):
        val = overrides.get(name, defaults.get(name, 0))
        if isinstance(val, torch.Tensor):
            params[0, :, i] = val[:T]
        else:
            params[0, :, i] = val
    return params


def play(params, normalize=True, label=None):
    """Synthesize and play audio."""
    with torch.no_grad():
        audio = pink_trombone(params)
    wav = audio[0]
    if normalize and wav.abs().max() > 0:
        wav = wav / wav.abs().max() * 0.9
    if label:
        print(label)
    display(Audio(wav.numpy(), rate=SAMPLE_RATE))
    return wav

## 1. Visualize the diameter profile

The tract is modeled as a tube of varying cross-sectional area.
The tongue controls shape the body; the constriction adds a local narrowing.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3), sharey=True)

configs = [
    dict(label="/ɑ/ (back open)",  tongue_idx=12.9, tongue_d=2.43, c_idx=25.0, c_d=2.0),
    dict(label="/i/ (front close)", tongue_idx=27.2, tongue_d=2.05, c_idx=25.0, c_d=2.0),
    dict(label="/u/ (back close)",  tongue_idx=22.8, tongue_d=2.05, c_idx=25.0, c_d=2.0),
]

for ax, cfg in zip(axes, configs):
    t_idx = torch.tensor([[cfg["tongue_idx"]]])
    t_d   = torch.tensor([[cfg["tongue_d"]]])
    c_idx = torch.tensor([[[cfg["c_idx"]]]])
    c_d   = torch.tensor([[[cfg["c_d"]]]])
    diam = _compute_diameter_profile(t_idx, t_d, c_idx, c_d)[0, 0,0].numpy()
    print(diam)
    print(diam.shape)
    ax.fill_between(range(_TRACT_N), diam, alpha=0.4)
    ax.plot(range(_TRACT_N), diam)
    ax.set_title(cfg["label"])
    ax.set_xlabel("Tract position (glottis → lips)")
    ax.set_ylim(0, 3)

axes[0].set_ylabel("Diameter")
plt.tight_layout()
plt.show()

## 2. Vowel sounds

Different tongue positions produce different vowels.

In [ ]:
vowels = {
    "/ɑ/ (father)": dict(tongueIndex=12.9, tongueDiameter=2.43),
    "/i/ (feet)": dict(tongueIndex=27.2, tongueDiameter=2.05),
    "/u/ (boot)": dict(tongueIndex=22.8, tongueDiameter=2.05),
    "/ɛ/ (bed)": dict(tongueIndex=20.0, tongueDiameter=2.3),
    "/ʌ/ (but)": dict(tongueIndex=17.0, tongueDiameter=2.3),
}

for label, kw in vowels.items():
    params = make_params(duration_s=1.5, frequency=140, tenseness=0.6, **kw)
    play(params, label=label)

## 3. Vowel glide (formant transition)

Sweep tongue index smoothly to produce a vowel glide (like "ee" → "ah").

In [ ]:
T = 38  # ~3 seconds
tongue_idx = torch.linspace(27.2, 12.9, T)  # /i/ → /ɑ/
tongue_d   = torch.linspace(2.05, 2.43, T)
params = make_params(duration_s=T / CONTROL_RATE,
                     tongueIndex=tongue_idx, tongueDiameter=tongue_d,
                     frequency=140, tenseness=0.6)
play(params, label="Glide /i/ → /ɑ/")

## 4. Constriction effects

`constrictionDiameter` narrows the tract at a chosen position.
- Near 2.0: no effect (open)
- Around 0.3–0.7: fricative turbulence noise
- Below 0: closed (stop consonant)

`constrictionIndex` sets the position (0 = glottis, 44 = lips).

In [ ]:
# Fricative at lip region (~35) - like /f/ or /v/
play(make_params(duration_s=2.0, frequency=140, tenseness=0.7,
                 constrictionIndex=35.0, constrictionDiameter=0.3),
     label="Fricative at lips (c_idx=35, c_diam=0.3)")

# Fricative mid-tract - like /s/
play(make_params(duration_s=2.0, frequency=140, tenseness=0.7,
                 constrictionIndex=32.0, constrictionDiameter=0.3),
     label="Fricative mid-tract (c_idx=32, c_diam=0.3)")

# Narrow constriction - approximant
play(make_params(duration_s=2.0, frequency=140, tenseness=0.6,
                 constrictionIndex=25.0, constrictionDiameter=0.8),
     label="Approximant (c_idx=25, c_diam=0.8)")

## 5. Constriction diameter sweep

Open → closed: from vowel to stop consonant.

In [ ]:
T = 38
c_diam = torch.linspace(2.0, -0.5, T)  # fully open → closed
params = make_params(duration_s=T / CONTROL_RATE,
                     frequency=140, tenseness=0.6,
                     tongueIndex=12.9, tongueDiameter=2.43,
                     constrictionIndex=30.0,
                     constrictionDiameter=c_diam)
play(params, label="Constriction sweep: open → closed")

## 6. Simple phrase: "ba" syllable

Close the tract (bilabial stop), then release into /ɑ/.

In [ ]:
T = 25  # 2s
# First 4 frames: closed bilabial stop; then open to /ɑ/
c_diam = torch.cat([torch.full((4,), -0.3), torch.linspace(-0.3, 2.0, 4), torch.full((T - 8,), 2.0)])
intensity = torch.cat([torch.zeros(4), torch.linspace(0, 1, 4), torch.ones(T - 8)])

params = make_params(duration_s=T / CONTROL_RATE,
                     frequency=140, tenseness=0.7,
                     tongueIndex=12.9, tongueDiameter=2.43,
                     constrictionIndex=40.0,  # lips
                     constrictionDiameter=c_diam,
                     intensity=intensity)
play(params, label='"ba" syllable')

## 7. Parameter gradients w.r.t. silence loss

Which parameters most affect the output energy?  
We compute gradients of MSE-from-silence over a 2-second clip.

In [ ]:
params = make_params(duration_s=2.0)
params.requires_grad_(True)

audio = pink_trombone(params)   # [1, S]
target = torch.zeros_like(audio)  # silence
loss = torch.nn.functional.mse_loss(audio, target)
loss.backward()

# Average absolute gradient across all frames
grad = params.grad[0]  # [T, N_PARAMS]
mean_abs_grad = grad.abs().mean(dim=0)  # [N_PARAMS]

print(f"Loss (MSE from silence): {loss.item():.6f}")
print()
print("Mean |∂loss/∂param| per parameter (averaged over control frames):")
for i, name in enumerate(PARAM_NAMES):
    bar = "█" * int(mean_abs_grad[i].item() / mean_abs_grad.max().item() * 30)
    print(f"  {name:>25s}: {mean_abs_grad[i].item():.2e}  {bar}")

In [ ]:
# Plot gradient magnitude per parameter
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(PARAM_NAMES, mean_abs_grad.detach().numpy())
ax.set_xlabel("|∂loss/∂param| (mean over control frames)")
ax.set_title("Parameter sensitivity (silence loss, 2s clip)")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 8. Gradient over time

Show how the gradient of key parameters evolves across control frames.

In [ ]:
interesting = ["frequency", "tenseness", "intensity", "tongueIndex", "constrictionDiameter"]
fig, axes = plt.subplots(len(interesting), 1, figsize=(10, 8), sharex=True)

for ax, name in zip(axes, interesting):
    i = PARAM_NAMES.index(name)
    g = grad[:, i].detach().numpy()
    ax.plot(g)
    ax.set_ylabel(name, fontsize=8)
    ax.axhline(0, color="k", linewidth=0.5)

axes[-1].set_xlabel("Control frame")
fig.suptitle("∂loss/∂param per control frame (silence loss)")
plt.tight_layout()
plt.show()